<h5>Import and clean data</h5>

In [18]:
import json
import pandas as pd
from src.normalization import get_normalized_list
from src.clean_data import clean_data

with open("data.json") as f:
    raw_data = json.load(f)

data = clean_data(raw_data)


# companies = pd.DataFrame(data["companies"])
# schools = pd.DataFrame(data["schools"])

normalized_companies = get_normalized_list(data["companies"])
normalized_schools = get_normalized_list(data["schools"])

# Add normalized company names to experiences
for experience in data["experiences"]:
    if experience["company"] is None:
        continue
    normalized_name = normalized_companies.get(experience["company"], experience["company"])
    experience["normalized_company"] = normalized_name

# Add normalized school names to educations
for education in data["educations"]:
    if education["school"] is None:
        continue
    normalized_name = normalized_schools.get(education["school"], education["school"])
    education["normalized_school"] = normalized_name

<h5>Top companies and schools</h5>

In [20]:
experiences_df = pd.DataFrame(data["experiences"])
educations_df = pd.DataFrame(data["educations"])

# What are the companies with the biggest amount of unique people
top_companies = (
    experiences_df
    .groupby("normalized_company")["person_id"]
    # .groupby("company")["person_id"]
    .nunique()
    .sort_values(ascending=False)
    .head(10)
)

# What are the schools with the biggest amount of unique people
top_schools = (
    educations_df
    .groupby("normalized_school")["person_id"]
    # .groupby("school")["person_id"]
    .nunique()
    .sort_values(ascending=False)
    .head(10)
)

# print(top_companies)
# print(top_schools)

<h5>Network</h5>

In [ ]:
import networkx as nx

G = nx.Graph()

people = pd.DataFrame(data["people"]).get("full_name")
companies = top_companies
schools = top_schools

print(top_companies)

# for person in people:
#     G.add_node(person, type="person")
for company in companies.index:
  G.add_node(company, type="company")
for school in schools.index:
    G.add_node(school, type="school")

nx.draw_spring(G, with_labels=True)

<h5>Bipartite Graph</h5>

In [ ]:
import matplotlib.pyplot as plt

filtered_df = experiences_df[
    experiences_df["company"].isin(top_companies.index)
]

B = nx.Graph()

for _, row in filtered_df.iterrows():
    person = row["person_id"]
    company = row["company"]

    B.add_node(person, bipartite=0)
    B.add_node(company, bipartite=1)

    B.add_edge(person, company)

nodes_people = {
    n for n, d in B.nodes(data=True)
    if d["bipartite"] == 0
}

nodes_companies = set(B) - nodes_people

pos = nx.bipartite_layout(
    B,
    nodes_people,
)

plt.figure(figsize=(16, 12))

nx.draw_networkx_nodes(
    B,
    pos,
    nodelist=nodes_people,
    node_size=50,
    alpha=0.6
)

nx.draw_networkx_nodes(
    B,
    pos,
    nodelist=nodes_companies,
    node_size=300,
    alpha=0.8
)

nx.draw_networkx_edges(
    B,
    pos,
    alpha=0.2
)

nx.draw_networkx_labels(
    B,
    pos,
    labels={c: c for c in nodes_companies},
    font_size=8
)

plt.savefig("graph.png")
plt.show()



<h5>Clustered Layout</h5>

In [ ]:
# Build bipartite graph from filtered experiences
filtered_df = experiences_df[experiences_df["company"].isin(top_companies.index)]
B = nx.Graph()
for _, row in filtered_df.iterrows():
    B.add_edge(row["person_id"], row["company"])

# Companies are the cluster "supernodes"
companies = list(top_companies.index)

# Create a supergraph (one node per company) to compute well-spaced centers
supergraph = nx.cycle_graph(len(companies))
superpos = nx.spring_layout(supergraph, scale=3, seed=42)
centers = list(superpos.values())

# For each company, compute a local layout for the company + its people
pos = {}
for center, company in zip(centers, companies):
    members = [company] + list(B.neighbors(company))
    if len(members) == 1:
        # company with no people — place at center
        pos[company] = tuple(center)
        continue
    subpos = nx.spring_layout(B.subgraph(members), center=center, seed=123)
    # Ensure the company stays at the exact center
    subpos[company] = tuple(center)
    pos.update(subpos)

# Draw
plt.figure(figsize=(14, 14))
nx.draw_networkx_edges(B, pos, alpha=0.08, width=0.6)

# People nodes: all nodes that are not company labels
people = [n for n in B.nodes() if n not in companies]
nx.draw_networkx_nodes(B, pos, nodelist=people, node_size=40, node_color="lightblue", alpha=0.7)

# Company nodes: larger and labeled
nx.draw_networkx_nodes(B, pos, nodelist=companies, node_size=900, node_color="coral", alpha=0.95)
nx.draw_networkx_labels(B, pos, labels={c: c for c in companies}, font_size=8)

plt.title("Companies as Cluster Centers with Their People")
plt.axis("off")
plt.tight_layout()
plt.savefig("graph.png", dpi=200)
plt.show()